In [9]:
# TIP: Use DataWrangler to view the DataFrame
query = 'RDF AND BIM AND reasoning'

In [10]:
import requests
import pandas as pd

# Retrieve all results using pagination (OpenAlex: max 200 per page)
all_records = []
page = 1
while True:
    paged_url = f'https://api.openalex.org/works?search={query}&per_page=200&page={page}'
    resp = requests.get(paged_url).json()
    # save response for debugging
    with open(f'response_page_{page}.json', 'w') as f:
        import json
        json.dump(resp, f, indent=2)
    records = resp.get('results', [])
    if not records:
        break
    all_records.extend(records)
    page += 1
    # Optional: stop after a certain number of pages to avoid long runtime
    if page > 5: break  # Remove or adjust as needed
dataframe = pd.DataFrame([{
    'cited_by_count': r['cited_by_count'],
    'year': r['publication_year'],
    'title': r['title'],
    'doi': r['doi'],
    'authors': [auth['author']['display_name'] for auth in r['authorships']],
    'relevance_score': r['relevance_score'],
    'abstract': r.get('abstract_inverted_index'),
} for r in all_records])
print(f'Total records retrieved: {len(dataframe)}')

Total records retrieved: 308


In [11]:
# Sort the DataFrame by most cited (descending by cited_by_count)
if 'cited_by_count' in dataframe.columns:
    dataframe = dataframe.sort_values(by='cited_by_count', ascending=False)
    print(dataframe[['title', 'cited_by_count', "doi"]].head(10))

                                                title  cited_by_count  \
3   Towards a semantic Construction Digital Twin: ...            1010   
9   Review of built heritage modelling: Integratio...             235   
6   BOT: The building topology ontology of the W3C...             222   
8                                               Brick             217   
22  Tendencies of Technologies and Platforms in Sm...             216   
7   Linking building data in the cloud: Integratin...             207   
11  Geospatial Data Management Research: Progress ...             161   
23  Big Data for Energy Management and Energy-Effi...             146   
10  Enhancing the ifcOWL ontology with an alternat...             109   
14       The promise of automated compliance checking             104   

                                             doi  
3   https://doi.org/10.1016/j.autcon.2020.103179  
9   https://doi.org/10.1016/j.culher.2020.05.008  
6              https://doi.org/10.3233/sw-2

In [12]:
# Count the number of papers per author and rank them
from collections import Counter

author_list = [author for authors in dataframe['authors'] for author in authors]
author_counts = Counter(author_list)
ranked_authors = author_counts.most_common()

# Display the top 10 authors with the most papers
for i, (author, count) in enumerate(ranked_authors[:20], 1):
    print(f"{i}. {author}: {count} papers")

1. Pieter Pauwels: 24 papers
2. Jakob Beetz: 7 papers
3. James O’Donnell: 6 papers
4. Yacine Rezgui: 4 papers
5. Edward Corry: 4 papers
6. Bauke de Vries: 4 papers
7. Roland Billen: 4 papers
8. Francesca Noardo: 4 papers
9. Mathias Bonduel: 4 papers
10. Haijiang Li: 4 papers
11. Peter Johansson: 4 papers
12. Seppo Törmä: 4 papers
13. Judith Michael: 4 papers
14. István Koren: 4 papers
15. Iraklis Dimitriadis: 4 papers
16. Judith Fulterer: 4 papers
17. Aymen Gannouni: 4 papers
18. Malte Heithoff: 4 papers
19. Annkristin Hermann: 4 papers
20. Katharina Hornberg: 4 papers
